In [17]:
!pip install transformers sentence-transformers faiss-cpu langchain langchain-text-splitters

In [18]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
with open("/content/drive/MyDrive/Colab Notebooks/my_knowledge.txt") as f:
  knowldege_text = f.read()

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 20,
    length_function = len
)

In [21]:
chunks = text_splitter.split_text(knowldege_text)

In [22]:
print(f"We have {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
  print(f"---Chunk {i+1}  ---\n{chunk}\n")

We have 5 chunks:
---Chunk 1  ---
Company Policy Manual:

---Chunk 2  ---
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

---Chunk 3  ---
Thursdays. Mondays and Fridays are optional remote days.

---Chunk 4  ---
- PTO Policy: Full-time employees receive 20 days of Paid Time Off (PTO) per year. PTO accrues monthly.

---Chunk 5  ---
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.



In [23]:
from sentence_transformers import SentenceTransformer

In [24]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [25]:
chunk_embeddings = model.encode(chunks)

In [26]:
print(f'shape of chunk_embeddings: {chunk_embeddings.shape}')

shape of chunk_embeddings: (5, 384)


In [27]:
import faiss
import numpy as np

In [28]:
d = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(np.array(chunk_embeddings).astype('float32'))
print(f'FAISS index created with {index.ntotal} vectors.')

FAISS index created with 5 vectors.


In [29]:
from transformers import pipeline
generator = pipeline("text2text-generation", model="google/flan-t5-small")

def question_answers(query):
  query_embedding=model.encode([query]).astype('float32')

  k=2
  distance,indices = index.search(query_embedding,k)

  retrieved_chunks = [chunks[i] for i in indices[0]]
  context = '\n\n'.join(retrieved_chunks)

  prompt_template = f"""
  Answer the following question using *only* the provided context.
  If the answer is not in the context, say "I don't have that information."

  Context:
  {context}

  Question:
  {query}

  Answer:
  """

  answer = generator(prompt_template,max_length=100)
  print(f'-----CONTEXT----\n{context}\n')
  return answer[0]['generated_text']


Device set to use cpu


In [30]:
query_1 ='What is the WFH policy?'
print(f'Query: {query_1}')
print(f'Answer: {question_answers(query_1)}\n')


Query: What is the WFH policy?


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-----CONTEXT----
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

Company Policy Manual:

Answer: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays Company Policy Manual:



In [31]:
query_2 = "What is the company's dental plan?"
print(f'Query: {query_2}')
print(f'Answer: {question_answers(query_2)}\n')

Query: What is the company's dental plan?


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-----CONTEXT----
Company Policy Manual:

- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

Answer: I don't have that information.



In [32]:
query_3 = "What is the tech stack required?"
print(f'Query: {query_3}')
print(f'Answer: {question_answers(query_3)}\n')

Query: What is the tech stack required?


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-----CONTEXT----
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.

Company Policy Manual:

Answer: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native. Company Policy Manual:

